# Cython 的应用

这一章说明 Cython 的进阶用法。

- 用扩展类型(`cdef class`)包住 C 的结构体
- 做一维的向量类型
- 包住 C 的库(multifast)
- 用 `@cython.ufunc` 做 ufunc
- 快速调用 DLL 的函数、直接调用 BLAS

In [1]:
import numpy as np
import helper.magics

%load_ext helper.cython

## 扩展类型

Python 的类把实例的属性放进字典，而内置类型把属性直接放在结构体的字段里，读写因此更快。

Cython 里用 `cdef class` 定义扩展类型：它同样把属性放在 C 的结构体里，从 Cython 内部访问因此快，很适合把 C 的函数库包成 Python 的对象来用。

### 扩展类型的写法

下面的程序用 `cdef class` 做出 `Point2D`，用 `cdef` 决定属性 `x`、`y`；与 Python 的类不同，属性在类里声明，而不是在 `__init__()` 里创建。

In [2]:
%%cython

cdef class Point2D:
    cdef public double x, y

Cython 会做出表示 `Point2D` 的下面这个结构体：`ob_refcnt` 与 `ob_type` 是任何 Python 对象都要的，C 里用 `PyObject_HEAD` 把它们合起来。

```cpp
struct __pyx_obj_Point2D {
  PyObject_HEAD
  double x;
  double y;
};
```

Cython 内部如果知道 `p` 是 `Point2D`，`p.x` 就会直接读结构体的字段。

In [3]:
print(type(Point2D.x))
print(Point2D.x.__get__)
print(Point2D.x.__set__)

<class 'getset_descriptor'>
<method-wrapper '__get__' of getset_descriptor object at 0x0000022A63512B00>
<method-wrapper '__set__' of getset_descriptor object at 0x0000022A63512B00>


方法与函数一样用 `def`、`cdef`、`cpdef` 定义：三种都能从 Cython 内部调用，但能从 Python 调用的只有 `def` 与 `cpdef`；在 Cython 内部调用 `cdef` 与 `cpdef` 会直接跑 C 的函数，比 `def` 快得多。

扩展类型可以继承扩展类型：下面 `Point3D` 继承 `Point2D`，再加上 `z`。

In [4]:
%%cython -a

cdef class Point2D:
    cdef public double x, y

cdef class Point3D(Point2D):
    cdef public double z
    
cdef Point3D p = Point3D()
p.x = 1.0
p.y = 2.0
p.z = 3.0

`Point3D` 的结构体如下。

```cpp
struct __pyx_obj_Point3D {
  struct __pyx_obj_Point2D __pyx_base;
  double z;
};
```

`__pyx_base` 就是父类 `Point2D` 的结构体：从 Cython 里用 `__pyx_base.x` 读父类的属性，从 Python 里则通过 `Point3D.__base__` 的描述符来读。

### 一维向量类型

以一维的 `Vector` 类型为例，看扩展类型怎么用：`Vector` 有两个内部属性，`count` 是长度、`data` 是数组开头的地址。

:::{note}
为了把较长的 Cython 代码分段给出，这里用本书的 `%include` 魔法命令。
:::

In [5]:
%include cython_examples/vector.pyx 1

```cython
cdef class Vector:
    cdef int count
    cdef double * data
```

做 `Vector` 时先要从堆里取出结构体的内存，再初始化属性，这个初始化由 `__cinit__()` 完成，它是 C 这一层的 `__init__()`。

`Vector` 支持两种造法：指定大小来申请，或者传入一个序列把内容填进去；这里用 `PyMem_Malloc()`，从 Python 管理的堆里取数组的内存。

In [6]:
%include cython_examples/vector.pyx 2

```cython
    def __cinit__(self, data):
        cdef int i
        if isinstance(data, int):
            self.count = data
        else:
            self.count = len(data)
        self.data = <double *>mem.PyMem_Malloc(sizeof(double)*self.count)
        if self.data is NULL:
            raise MemoryError
        
        if not isinstance(data, int):
            for i in range(self.count):
                self.data[i] = data[i]
```

引用计数减到 0 时，回收之前会调用 `__dealloc__()`，在那里释放 `data` 的内存。

In [7]:
%include cython_examples/vector.pyx 3

```cython
    def __dealloc__(self):
        if self.data is not NULL:
            mem.PyMem_Free(self.data)
```

要让 `Vector` 能用下标与循环，就定义 `__len__()`、`__getitem__()`、`__setitem__()`：前后各两个下划线的方法叫魔法方法，它们决定各种语法的动作，`len(obj)` 会调 `obj.__len__()`，`obj[i]` 会调 `obj.__getitem__(i)`。

In [8]:
%include cython_examples/vector.pyx 4

```cython
    def __len__(self):
        return self.count
    
    cdef _check_index(self, int *index):
        if index[0] < 0:
            index[0] = self.count + index[0]
        if index[0] < 0  or index[0] > self.count - 1:
            raise IndexError("Vector index out of range")
    
    def __getitem__(self, int index):
        self._check_index(&index)
        return self.data[index]
    
    def __setitem__(self, int index, double value):
        self._check_index(&index)
        self.data[index] = value
```

要让加法可用，就定义 `__add__()`。

In [9]:
%include cython_examples/vector.pyx 5

```cython
    def _add(self, other):        
        cdef Vector new, _other

        if isinstance(other, Vector): #❷       
            _other = <Vector>other #❸
            if self.count != _other.count:
                raise ValueError("Vector size not equal")
            new = Vector(self.count) #❹
            add_array(self.data, _other.data, new.data, self.count)
            return new
        new = Vector(self.count) #❹
        add_number(self.data, <double>other, new.data, self.count)
        return new

    def __add__(self, other): #❶
        return self._add(other)

    def __radd__(self, other): #❶
        return self._add(other)
```

二项运算的方法要考虑左右两种情形：`a + b` 先调 `a.__add__(b)`，失败之后再调 `b.__radd__(a)`。

`v` 是 `Vector` 时，`1 + v` 里 `int` 的 `__add__()` 会失败，于是调用 `Vector.__radd__()`。

In [10]:
%include cython_examples/vector.pyx 6

```cython
    def __iadd__(self, other):
        cdef Vector _other
        if isinstance(other, Vector):
            _other = <Vector>other
            if self.count != _other.count:
                raise ValueError("Vector size not equal")
            add_array(self.data, _other.data, self.data, self.count)
        else:
            add_number(self.data, <double>other, self.data, self.count)
        return self
```

`__iadd__()` 的第 1 个参数就是自己，Cython 因此知道类型，不需要转换。

下面的代码用 `cpdef` 定义 `norm()`，再从 `__str__()` 调用它：`cpdef` 会同时做出 Cython 与 Python 两个入口，而 `__str__()` 是从 Cython 的入口调用 `norm()`。

In [11]:
%include cython_examples/vector.pyx 7

```cython
    def __str__(self):
        values = ", ".join(str(self.data[i]) for i in range(self.count))
        norm = self.norm()
        return "Vector[{}]({})".format(norm, values)
    
    cpdef norm(self):
        cdef double *p
        cdef double s
        cdef int i
        s = 0
        p = self.data
        for i in range(self.count):
            s += p[i] * p[i]
        return s**0.5
```

扩展类型也能从 Python 继承，并覆盖 `def` 与 `cpdef` 的方法，被覆盖的方法从 Cython 调用时也能正确走到；不过 `cpdef` 会检查是否被覆盖，比 `cdef` 稍慢。

最后在开头加上 `add_array()` 与 `add_number()`，两个都是 `cdef`，只能从 Cython 内部调用。

In [12]:
%include cython_examples/vector.pyx 8

```cython
cdef add_array(double *op1, double *op2, double *res, int count):
    cdef int i
    for i in range(count):
        res[i] = op1[i] + op2[i]

cdef add_number(double *op1, double op2, double *res, int count):
    cdef int i
    for i in range(count):
        res[i] = op1[i] + op2
```

In [13]:
import subprocess
import sys

# 扩展模块不用 PATH 上的 cythonize，而是用内核的 Python 来做。
# 为了让 setup.cfg(mingw32) 被读到，在 cython_examples 里执行。
build = subprocess.run(
    [sys.executable, "setup.py", "build_ext", "--inplace"],
    cwd="cython_examples", capture_output=True, text=True,
)
print(build.stdout[-600:] or build.stderr[-600:])

其他的二项运算与元素访问，也请照同样方式写写看，下面是 `Vector` 的使用例。

In [14]:
from cython_examples.vector import Vector

v1 = Vector(range(5))
v2 = Vector(range(100, 105))
print(len(v1))
print(v1 + v2)
print(v1 + 2)
print(20 + v2)
print(v1.norm(), v2.norm())
print([x**2 for x in v1])

5
Vector[232.63705637752557](100.0, 102.0, 104.0, 106.0, 108.0)
Vector[9.486832980505138](2.0, 3.0, 4.0, 5.0, 6.0)
Vector[272.81862106535175](120.0, 121.0, 122.0, 123.0, 124.0)
5.477225575051661 228.10085488660502
[0.0, 1.0, 4.0, 9.0, 16.0]


比较 `Vector` 与 NumPy 数组做向量加法的速度。

In [15]:
v1 = Vector(range(10000))
v2 = Vector(range(10000))
%timeit v1 + v2

a1 = np.arange(10000, dtype=float)
a2 = np.arange(10000, dtype=float)
%timeit a1 + a2

7.37 μs ± 324 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


6.78 μs ± 119 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


也比较元素访问的速度。

In [16]:
%timeit v1[100]
%timeit v1[100] = 2.0
%timeit a1[100]
%timeit a1[100] = 2.0

54.8 ns ± 2 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


44.4 ns ± 0.796 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


121 ns ± 3.71 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


107 ns ± 3.56 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


### 包住 multifast

扩展类型也用来把 C 的函数库包成 Python 的对象，这里包住同时查找多个模式的 C 库 `multifast`。

:::{seealso}
http://multifast.sourceforge.net/v1_4_2/

实现多模式匹配的 multifast 项目
:::

这是做好的扩展类型的用法：把关键字排起来做成 `MultiSearch`，再用 `isin()` 查目标字节串里是否含有其中之一，含有就是 `True`，没有就是 `False`。

In [17]:
from cython_examples.multisearch import MultiSearch

ms = MultiSearch([b"abc", b"xyz"])
print(ms.isin(b"123abcdef"))
print(ms.isin(b"123uvwxyz"))
print(ms.isin(b"123456789"))

True
True
False


`search()` 找出关键字的位置：第 2 个参数是个函数，每找到一个就把位置与关键字传过去，返回 0 就继续、返回 1 就停止。

In [18]:
def process(pos, pattern):
    print("found {0} at {1}".format(pattern, pos))
    return 0


ms.search(b"123abc456xyz789abc", process)

found b'abc' at 3
found b'xyz' at 9
found b'abc' at 15


用 `iter_search()` 则得到一个依次返回找到的内容的迭代器。

In [19]:
for pos, pattern in ms.iter_search(b"123abc456xyz789abc"):
    print("found {0} at {1}".format(pattern, pos))

found b'abc' at 3
found b'xyz' at 9
found b'abc' at 15


写扩展类型之前，先看 C 里怎样使用这个库。

```c
#include <stdio.h>
#include "ahocorasick.h"

/* 搜索关键字列表 */
AC_ALPHABET_t * allstr[] = {
    "recent", "from", "college"
};

#define PATTERN_NUMBER (sizeof(allstr)/sizeof(AC_ALPHABET_t *))

/* 搜索的文本 */
AC_ALPHABET_t * input_text = {"She recently graduated from college"};

//*** 匹配时的回调函数
int match_handler(AC_MATCH_t * m, void * param)
{
    unsigned int j;

    printf ("@ %ld : %s
", m->position, m->patterns->astring);
    /* 返回 0 继续搜索，返回 1 停止搜索 */
    return 0;
}

int main (int argc, char ** argv)
{
    unsigned int i;

    AC_AUTOMATA_t * acap;
    AC_PATTERN_t tmp_patt;
    AC_TEXT_t tmp_text;

    //*** 做出 AC_AUTOMATA_t 结构体，并传入回调函数
    acap = ac_automata_init();

    //*** 添加关键字
    for (i=0; i<PATTERN_NUMBER; i++)
    {
        tmp_patt.astring = allstr[i];
        tmp_patt.rep.number = i+1; // optional
        tmp_patt.length = strlen(tmp_patt.astring);
        ac_automata_add (acap, &tmp_patt);
    }

    //*** 结束关键字的添加
    ac_automata_finalize (acap);

    //*** 设置要搜索的字节串
    tmp_text.astring = input_text;
    tmp_text.length = strlen(tmp_text.astring);

    //*** 执行搜索
    ac_automata_search (acap, &tmp_text, 0, match_handler, NULL);

    //*** 释放内存
    ac_automata_release (acap);
    return 0;
}
```

这个库以 `AC_AUTOMATA_t` 结构体为中心，这是 C 里把数据汇总起来的常见做法。

用 Cython 包它时，把这个结构体的指针做成属性，在 `__cinit__()` 里申请、在 `__dealloc__()` 里释放，然后再排上调用 C API 的 `def` 方法。

In [20]:
%include cython_examples/multisearch.pyx 1

```cython
cdef extern from "ahocorasick.h": #❶
    ctypedef int (*AC_MATCH_CALBACK_f)(AC_MATCH_t *, void *) #❷
    ctypedef enum AC_STATUS_t: #❸
        ACERR_SUCCESS = 0
        ACERR_DUPLICATE_PATTERN
        ACERR_LONG_PATTERN
        ACERR_ZERO_PATTERN
        ACERR_AUTOMATA_CLOSED

    ctypedef struct AC_MATCH_t: #❹
        AC_PATTERN_t * patterns
        long position
        unsigned int match_num

    ctypedef struct AC_AUTOMATA_t:
        AC_MATCH_t match

    ctypedef struct AC_PATTERN_t:
        char * astring
        unsigned int length

    ctypedef struct AC_TEXT_t:
        char * astring
        unsigned int length

    #❺
    AC_AUTOMATA_t * ac_automata_init() 
    AC_STATUS_t ac_automata_add(AC_AUTOMATA_t * thiz, AC_PATTERN_t * pattern)
    void ac_automata_finalize(AC_AUTOMATA_t * thiz)
    int ac_automata_search(AC_AUTOMATA_t * thiz, AC_TEXT_t * text, int keep, 
        AC_MATCH_CALBACK_f callback, void * param)
    void ac_automata_settext (AC_AUTOMATA_t * thiz, AC_TEXT_t * text, int keep)
    AC_MATCH_t * ac_automata_findnext (AC_AUTOMATA_t * thiz)        
    void ac_automata_release(AC_AUTOMATA_t * thiz)
```

❶ 先用 `cdef extern from ...` 让编译后的 C 去读 `ahocorasick.h`；Cython 不会自动读 C 的头文件，因此要用 Cython 的语法声明用到的类型、常量与函数。

❷ 定义函数指针类型 `MATCH_CALLBACK_f`，第 1 个参数是匹配的信息、第 2 个参数是指向任意数据的指针；C 里用 `void *` 什么都能传。

In [21]:
%include cython_examples/multisearch.pyx 2

```cython
cdef class MultiSearch:
    
    cdef AC_AUTOMATA_t * _auto #❶
    cdef bint found
    cdef object callback
    cdef object exc_info
    cdef list _keywords

    def __cinit__(self, keywords):
        self._auto = ac_automata_init()
        if self._auto is NULL:
            raise MemoryError
        self._keywords = []
        self.add(keywords) #❷

    def __dealloc__(self):
        if self._auto is not NULL:
            ac_automata_release(self._auto)
            
    cdef add(self, keywords):
        cdef AC_PATTERN_t pattern
        cdef bytes keyword
        cdef AC_STATUS_t err
        
        for keyword in keywords: #❸
            self._keywords.append(keyword)
            pattern.astring = <char *>keyword
            pattern.length = len(keyword)
            err = ac_automata_add(self._auto, &pattern)
            if err != ACERR_SUCCESS:
                raise ValueError("Error Code:%d" % err)

        ac_automata_finalize(self._auto)
```

❶ `_auto` 是指向 `AC_AUTOMATA_t` 的指针：在 `__cinit__()` 里调用 `ac_automata_init()` 申请，在 `__dealloc__()` 里调用 `ac_automata_release()` 释放。❷ 申请成功之后，用 `cdef` 的 `add()` 把所有关键字加进去。

❸ `add()` 里遍历 `keywords`，把每个元素当作 `bytes`，再用 `<char *>` 变成 C 的指针。

In [22]:
%include cython_examples/multisearch.pyx 3

```cython
    def isin(self, bytes text, bint keep=False):
        cdef AC_TEXT_t temp_text   #❶
        temp_text.astring = <char *>text
        temp_text.length = len(text)
        self.found = False         #❷
        ac_automata_search(self._auto, &temp_text, keep, isin_callback, <void *>self) #❸
        return self.found
```

❶ 把目标的字节串包进 `AC_TEXT_t`，❸ 再交给 `ac_automata_search()` 去找；在那之前先 ❷ 把 `found` 设成 `False`。传进 `isin_callback()` 的地址，让它在找到时被调用，最后一个参数是传给那个函数的数据，这里是 `MultiSearch` 的地址，这样函数内部就能访问对象的属性。

In [23]:
%include cython_examples/multisearch.pyx 4

```cython
cdef int isin_callback(AC_MATCH_t * match, void * param) noexcept:
    cdef MultiSearch ms = <MultiSearch> param #❶
    ms.found = True  #❷
    return 1  #❸
```

`isin_callback()` 的第 1 个参数是匹配的信息，第 2 个参数是指向 `MultiSearch` 的指针。

❶ 先把 `void *` 还原成 `MultiSearch`，再用 `ms` 访问属性与方法，❷ 把 `found` 设成 `True`。❸ `isin()` 找到一个就够了，于是返回 1 停下来。

:::{tip}
这里用 `ac_automata_search()` 是为了展示 C 与 Python 函数的用法；实际写的时候用后面出现的 `ac_automata_findnext()`，`isin()` 与 `search()` 都会简单得多。
:::

In [24]:
%include cython_examples/multisearch.pyx 5

```cython
    def search(self, bytes text, callback, bint keep=False):
        cdef AC_TEXT_t temp_text
        temp_text.astring = <char *>text
        temp_text.length = len(text)
        self.found = False
        self.callback = callback  #❶
        self.exc_info = None
        ac_automata_search(self._auto, &temp_text, keep, search_callback, <void *>self) #❷
        if self.exc_info is not None:
            raise self.exc_info[1], None, self.exc_info[2]  #❸
```

❷ `ac_automata_search()` 要调的函数是 `search_callback()`。❶ 先把 `callback` 存进 `self.callback`，❸ C 的函数没法把 Python 的异常往上抛，先把错误记在 `exc_info` 里，之后再抛。

In [25]:
%include cython_examples/multisearch.pyx 6

```cython
cdef int search_callback(AC_MATCH_t * match, void * param) noexcept:
    cdef MultiSearch ms = <MultiSearch> param
    cdef bytes pattern = match.patterns.astring
    cdef int res = 1
    try:
        res = ms.callback(match.position - len(pattern), pattern)  #❶
    except Exception as ex:
        import sys
        ms.exc_info = sys.exc_info()  #❷
    return res
```

把第 2 个参数还原成 `MultiSearch` 之后，❶ 调用 `callback` 里的 Python 函数，❷ 出了异常就记进 `exc_info`；`search()` 的最后检查 `exc_info`，有的话就抛出来。用 `sys.exc_info()` 是为了留下正确的 traceback。

multifast 里还有 `ac_automata_settext()` 与 `ac_automata_findnext()`，用这两个就能做出 `iter_search()`。

In [26]:
%include cython_examples/multisearch.pyx 7

```cython
    def iter_search(self, bytes text, bint keep=False):
        cdef AC_TEXT_t temp_text
        cdef AC_MATCH_t * match
        cdef bytes matched_pattern
        temp_text.astring = <char *>text
        temp_text.length = len(text)
        ac_automata_settext(self._auto, &temp_text, keep)
        while True:
            match = ac_automata_findnext(self._auto)
            if match == NULL:
                break
            matched_pattern = <bytes>match.patterns.astring
            yield match.position - len(matched_pattern), matched_pattern
```

## Cython 的技巧

到前面为止，用 Cython 提速的基本做法应该已经清楚了，最后再看几个进阶的用法。

### 做 ufunc

NumPy 的 ufunc 是对数组的每个元素做同样计算的函数，用 C 的 API 自己也能做。

用 `@cython.ufunc` 就能把普通的函数变成 ufunc，下面的程序做出计算逻辑斯谛函数 $\frac{1}{1+e^{-x}}$ 的 ufunc `logistic()`。

:::{seealso}
[用 NumPy 的 C-API 做 ufunc 的教程](https://numpy.org/doc/stable/user/c-info.ufunc-tutorial.html)
:::

In [27]:
%%cython --compile-args=-w
# distutils: define_macros=NPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION
cimport cython
from libc.math cimport exp

@cython.ufunc
cdef double logistic(double x):
    return 1.0 / (1.0 + exp(-x))

试一下 `logistic()`：虽然只写了 `double` 的计算，却能传各种类型的数组与列表。

In [28]:
logistic([-1, 0, 1])

array([0.26894142, 0.5       , 0.73105858])

传入 `float32` 的数组，结果也是 `float64`，因为 `logistic()` 的输入输出只有 `double`。

In [29]:
logistic(np.arange(-1, 2, dtype=np.float32))

array([0.26894142, 0.5       , 0.73105858])

要同时支持 `float32` 与 `float64`，就声明融合类型 `float_num`，把输入输出都用这个类型。

In [30]:
%%cython --compile-args=-w
# distutils: define_macros=NPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION
cimport cython
from libc.math cimport exp

ctypedef fused float_num:
    double
    float

@cython.ufunc
cdef float_num logistic2(float_num x):
    return 1.0 / (1.0 + exp(-x))

传整数得到 `float64`，传 `float32` 得到 `float32`。

In [31]:
print(logistic2([-1, 0, 1]).dtype)
print(logistic2(np.arange(-1, 2, dtype=np.float32)).dtype)

float64
float32


也能做参数不止一个的函数，下面 `peaks()` 就是双变量的函数。

In [32]:
%%cython --compile-args=-w
# distutils: define_macros=NPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION
cimport cython
from libc.math cimport exp


@cython.ufunc
cdef double peaks(double x, double y):
    return x * exp(-x*x - y*y)

In [33]:
Y, X = np.ogrid[-2:2:5j, -2:2:5j]
peaks(X, Y)

array([[-0.00067093, -0.00673795,  0.        ,  0.00673795,  0.00067093],
       [-0.01347589, -0.13533528,  0.        ,  0.13533528,  0.01347589],
       [-0.03663128, -0.36787944,  0.        ,  0.36787944,  0.03663128],
       [-0.01347589, -0.13533528,  0.        ,  0.13533528,  0.01347589],
       [-0.00067093, -0.00673795,  0.        ,  0.00673795,  0.00067093]])

返回值有多个时，就用元组来声明类型：下面的例子是返回二次方程两个解的 ufunc，参数是 `double`，返回值是 2 个 `complex`。

In [34]:
%%cython --compile-args=-w
# distutils: define_macros=NPY_NO_DEPRECATED_API=NPY_1_7_API_VERSION
cimport cython
from libc.complex cimport csqrt

@cython.ufunc
cdef (complex, complex) quadratic_roots(double a, double b, double c):
    cdef complex t0, t1
    t0 = csqrt(-4*a*c + b**2)
    t1 = 0.5 / a
    return t1*(-b - t0), t1*(-b + t0)

In [35]:
a, b, c = np.ogrid[-1:1:4j, -1:1:3j, -1:1:2j]
x0, x1 = quadratic_roots(a, b, c)
print(x0.shape)
print(x1.shape)
print(np.allclose(a * x0 ** 2 + b * x0 + c, 0))
print(np.allclose(a * x1 ** 2 + b * x1 + c, 0))

(4, 3, 2)
(4, 3, 2)
True
True


### 快速调用 DLL 的函数

用 `ctypes` 或 `cffi` 也能调用 DLL 的函数，但调用之前的准备工作不少，因此不快；在循环里反复调用时，这些准备工作的费用就显出来了。

这里用 `cffi` 取到函数的地址，把地址交给 Cython，再从 Cython 的循环里调用。

In [36]:
%%writefile peaks.c
#include <math.h>
double peaks(double x, double y)
{
    return x * exp(-x*x - y*y);
}

unsigned long long get_addr()
{
    return (unsigned long long)(void *)peaks;
}

Writing peaks.c


用 `gcc` 把 `peaks.c` 编译成 `peaks.dll`。

In [37]:
!gcc -Ofast -shared -o peaks.dll peaks.c

用 `cffi` 读入 `peaks.dll`，决定函数的类型再调用。

In [38]:
import cffi

ffi = cffi.FFI()
ffi.cdef(
    """
unsigned long long get_addr();
double peaks(double x, double y);
"""
)
lib = ffi.dlopen("peaks.dll")
lib.peaks(1.0, 2.0)

0.006737946999085467

`lib.peaks` 是包着 C 函数地址的 `cffi` 对象：用 `ffi.addressof()` 能取到指针，用 `ffi.cast()` 能变成整数。下面的程序把这个值与 `get_addr()` 的返回值相比较。

In [39]:
peak_addr = ffi.cast("size_t", ffi.addressof(lib, "peaks"))
assert peak_addr == lib.get_addr()

在 Cython 里做 `vectorize_2d()`：`func_addr` 是函数的地址，`x` 与 `y` 是二维的连续数组。

❶ 用 `ctypedef` 声明接受两个 `double`、返回 `double` 的函数指针类型 `Function`。❷ 把 `func_addr` 转换成 `Function`，在二重循环里调用那个函数。

In [40]:
%%cython
import cython
import numpy as np

ctypedef double(*Function)(double x, double y)      #❶

@cython.wraparound(False)
@cython.boundscheck(False)
def vectorize_2d(size_t func_addr, double[:, ::1] x, double[:, ::1] y):
    cdef double[:, ::1] res = np.zeros_like(x.base)
    cdef Function func_ptr = <Function><void *>func_addr #❷
    cdef int i, j
    
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            res[i, j] = func_ptr(x[i, j], y[i, j])
            
    return res.base

用 `vectorize_2d()` 调用 `peaks()`，再与 `vectorize()` 做出来的 ufunc 的结果比较。

In [41]:
Y, X = np.mgrid[-2:2:200j, -2:2:200j]
vectorize_peaks = np.vectorize(lib.peaks, otypes=["f8"])
np.allclose(vectorize_peaks(X, Y), vectorize_2d(peak_addr, X, Y))

True

用本书的 `FuncAddr`，就能方便地取到 DLL 函数的地址。

In [42]:
from helper.cffi import FuncAddr

msvcrt = FuncAddr("msvcrt.dll")
np.allclose(np.arctan2(X, Y), vectorize_2d(msvcrt.atan2, X, Y))

True

### 调用 BLAS

BLAS 是线性代数基本例程的标准，SciPy 里快速的线性代数就是调用其中的 Fortran BLAS；Fortran 的函数本身很快，但从 Python 调用有费用，放进循环就成了问题，而从 Cython 的循环里调用，这笔费用就没了。

#### 包住 `saxpy()`

In [43]:
from scipy.linalg import blas
import numpy as np

x = np.array([1, 2, 3], np.float32)
y = np.array([1, 3, 5], np.float32)
print(blas.saxpy)
blas.saxpy(x, y, a=0.5)

<fortran function saxpy>


array([1.5, 4. , 6.5], dtype=float32)

BLAS 的函数可以从 `scipy.linalg.blas` 用，下面是 `saxpy()` 的例子。

`saxpy` 是 `<fortran object>`，每次调用都要转换 Python 的对象，不适合放进循环。

`scipy.linalg` 里还有给 Cython 调用的 `cython_blas`，`saxpy()` 的类型如下。

```cython
ctypedef float s
cdef void saxpy(int *n, s *sa, s *sx, int *incx, s *sy, int *incy) nogil
```

In [44]:
%%cython
import cython
cimport scipy.linalg.cython_blas as blas

def blas_saxpy(float[:] y, float a, float[:] x):
    cdef int n = y.shape[0]
    cdef int inc_x = x.strides[0] // sizeof(float)
    cdef int inc_y = y.strides[0] // sizeof(float)
    blas.saxpy(&n, &a, &x[0], &inc_x, &y[0], &inc_y)
    
@cython.wraparound(False)
@cython.boundscheck(False)
def cython_saxpy(float[:] y, float a, float[:] x):
    cdef int i
    for i in range(y.shape[0]):
        y[i] += a * x[i]

比较两种速度：从 Cython 的循环里调用更快。

In [45]:
a = np.arange(100000, dtype=np.float32)
b = np.zeros_like(a)
%timeit blas_saxpy(b, 0.2, a)
%timeit cython_saxpy(b, 0.2, a)

20.1 μs ± 423 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


97.2 μs ± 3.51 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


#### 用 `dgemm()` 做矩阵积

BLAS 的 `DGEMM()` 计算下面的矩阵积，`alpha` 为 1、`beta` 为 0 时 `C` 就是 `A` 与 `B` 的积。

```
C = alpha*op(A)*op(B) + beta*C
```

`op()` 可以表示转置。Fortran 的数组与 C 的数组轴序相反，因此 C 的数组之间相乘时要把 `op()` 设成转置；即使转置，结果 `C` 仍然是 Fortran 的顺序。

下面的 `dgemm(A, B, index)` 里，`A` 与 `B` 是 C 顺序的三维数组，形状分别是 `(La, M, K)` 与 `(Lb, K, N)`；`index` 是形状 `(Lc, 2)` 的整数数组，对每一组 `j`、`k` 计算 `C[i] = A[j] * B[k]`。返回值 `C` 的形状是 `(Lc, N, M)`，可以看作 `Lc` 个 Fortran 的二维数组。

In [46]:
%%cython -c-Ofast -c-fopenmp --link-args=-fopenmp

from cython.parallel import prange #❶
import cython
import numpy as np
cimport scipy.linalg.cython_blas as blas

@cython.wraparound(False)
@cython.boundscheck(False)
def dgemm(double[:, :, :] A, double[:, :, :] B, int[:, ::1] index):
    cdef int m, n, k, i, length, idx_a, idx_b
    cdef double[:, :, :] C
    cdef char ta, tb
    cdef double alpha = 1.0
    cdef double beta = 0.0
        
    length = index.shape[0]
    m, k, n  = A.shape[1], A.shape[2], B.shape[2]        
    
    C = np.zeros((length, n, m))
    
    ta = b"T"
    tb = ta
    
    for i in prange(length, nogil=True): #❷
        idx_a = index[i, 0]
        idx_b = index[i, 1]
        blas.dgemm(&ta, &tb, &m, &n, &k, &alpha, 
               &A[idx_a, 0, 0], &k, 
               &B[idx_b, 0, 0], &n, 
               &beta,
               &C[i, 0, 0], &m)
    
    return C.base

NumPy 的 gufunc 能把一个矩阵的计算扩展到整个数组，`@` 运算符也一样，用 `dgemm()` 也能做到。

下面 `matrix_multiply(a, b)` 在最后两个轴上算矩阵积，其余的轴则广播。

In [47]:
def matrix_multiply(a, b):
    if a.ndim <= 2 and b.ndim <= 2:
        return np.dot(a, b)

    a = np.ascontiguousarray(a).astype(np.float64, copy=False)
    b = np.ascontiguousarray(b).astype(np.float64, copy=False)
    if a.ndim == 2:
        a = a[None, :, :]
    if b.ndim == 2:
        b = b[None, :, :]

    shape_a = a.shape[:-2]
    shape_b = b.shape[:-2]
    len_a = np.prod(shape_a)
    len_b = np.prod(shape_b)

    idx_a = np.arange(len_a, dtype=np.int32).reshape(shape_a)
    idx_b = np.arange(len_b, dtype=np.int32).reshape(shape_b)
    idx_a, idx_b = np.broadcast_arrays(idx_a, idx_b)

    index = np.column_stack((idx_a.ravel(), idx_b.ravel()))
    bshape = idx_a.shape

    if a.ndim > 3:
        a = a.reshape(-1, a.shape[-2], a.shape[-1])
    if b.ndim > 3:
        b = b.reshape(-1, b.shape[-2], b.shape[-1])

    if a.shape[-1] != b.shape[-2]:
        raise ValueError("can't do matrix multiply because k isn't the same")

    c = dgemm(a, b, index)
    c = np.swapaxes(c, -2, -1)
    c.shape = bshape + c.shape[-2:]
    return c

把 `matrix_multiply()` 与 `@` 的结果相比较。

In [48]:
a = np.random.rand(12, 1, 10, 100, 30)
b = np.random.rand(1, 15, 1, 30, 50)
np.allclose(matrix_multiply(a, b), a @ b)

C:\Users\ruoyu\AppData\Local\Temp\ipykernel_29712\1110928923.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  c.shape = bshape + c.shape[-2:]


True

也比较速度：`matrix_multiply()` 更快。

In [49]:
%timeit matrix_multiply(a, b)
%timeit a @ b

C:\Users\ruoyu\AppData\Local\Temp\ipykernel_29712\1110928923.py:34: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  c.shape = bshape + c.shape[-2:]


20.7 ms ± 1.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


66.7 ms ± 1.22 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## 小结

* 用 `cdef class` 做扩展类型，属性放在结构体里，因此快。
* 属性在类里声明，用 `__cinit__()` 初始化、用 `__dealloc__()` 释放。
* 用魔法方法(`__len__`、`__getitem__`、`__add__`、`__radd__` 等)支撑各种语法。
* 包 C 的库时，把结构体的指针做成属性，用 `cdef extern from` 声明类型与函数。
* 要把 Python 的异常传过 C 的回调，就记进 `exc_info` 再抛出来。
* 用 `@cython.ufunc` 能做 ufunc，类型可以用融合类型扩展。
* DLL 的函数取到地址，在 Cython 里变成函数指针再调用就很快。
* BLAS 从 `cython_blas` 调用就没有 Python 调用的费用，用 `dgemm()` 还能一次算一批矩阵积。